In [1]:
import numpy as np 
import pandas as pd 
import os 
import scanpy as sc 


In [4]:
reg = pd.read_csv('/home/rsun@ZHANGroup.local/sly_data/grn_momics/scenic_hm_outs/scenic_pipe/Snakemake/eRegulon_direct.tsv', sep='\t')
print(reg.shape)
reg.head()

(25257, 15)


,Region,Gene,importance_R2G,rho_R2G,importance_x_rho,importance_x_abs_rho,TF,is_extended,eRegulon_name,Gene_signature_name,Region_signature_name,importance_TF2G,regulation,rho_TF2G,triplet_rank
0,chr11:120280448-120280948,Notum,0.006594,0.050808,0.000335,0.000335,Arx,False,Arx_direct_+/+,Arx_direct_+/+_(8g),Arx_direct_+/+_(11r),1.080776,1,0.081685,4682
1,chrX:162448974-162449474,Ap1s2,0.021331,0.080816,0.001724,0.001724,Arx,False,Arx_direct_+/+,Arx_direct_+/+_(8g),Arx_direct_+/+_(11r),1.182803,1,0.056443,15833
2,chr10:42583455-42583955,Nr2e1,0.042814,0.101839,0.004360,0.004360,Arx,False,Arx_direct_+/+,Arx_direct_+/+_(8g),Arx_direct_+/+_(11r),0.997486,1,0.109166,3203
3,chr11:101162530-101163030,Etv4,0.016950,0.058818,0.000997,0.000997,Arx,False,Arx_direct_+/+,Arx_direct_+/+_(8g),Arx_direct_+/+_(11r),2.917420,1,0.074825,4148
4,chr6:114187952-114188452,Slc6a1,0.043789,0.187216,0.008198,0.008198,Arx,False,Arx_direct_+/+,Arx_direct_+/+_(8g),Arx_direct_+/+_(11r),4.502948,1,0.197486,402


In [ ]:

# tf_list = ['Sox11', 'Cebpb']
# idx = reg['TF'].isin(tf_list)
# reg = reg.loc[idx,:]
# print(reg['Gene'].nunique(), reg['Region'].nunique())

67 74


In [5]:
reg_list = ['Cebpb_direct_+/+', 'Sox11_direct_-/-']
idx = reg['eRegulon_name'].isin(reg_list)
reg = reg.loc[idx,:]
print(reg['Gene'].nunique(), reg['Region'].nunique(), reg.shape)

36 41 (42, 15)


In [6]:
tg_list = reg['Gene'].unique()
re_list = reg['Region'].unique()
reg_list = []
for i in range(reg.shape[0]):
    reg_list.append(f"{reg['TF'].iloc[i]}|{reg['Region'].iloc[i]}|{reg['Gene'].iloc[i]}")
reg_list = np.array(reg_list)

## 获取 tf,tg expression, re openness

In [7]:
import mudata
adata = mudata.read('/home/rsun@ZHANGroup.local/sly_data/grn_momics/scenic_hm_outs/scenic_pipe/Snakemake/ACC_GEX.h5mu')
adata

/home/rsun@ZHANGroup.local/anaconda3/envs/scenicplus/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/rsun@ZHANGroup.local/anaconda3/envs/scenicplus/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


MuData object with n_obs × n_vars = 19409 × 262002
  2 modalities
    scRNA:	19409 x 9911
      obs:	'leiden', 'eco', 'species', 'sample'
      var:	'gene_ids', 'feature_types', 'genome'
      obsm:	'X_pca', 'X_pca_harmony', 'X_umap'
    scATAC:	19409 x 252091
      obs:	'sample', 'cisTopic_log_nr_acc', 'eco', 'cisTopic_nr_acc', 'barcode_rank', 'pdf_values_for_tss_enrichment', 'tss_enrichment', 'barcode', 'unique_fragments_in_peaks_count', 'log10_total_fragments_in_peaks_count', 'total_fragments_in_peaks_count', 'total_fragments_count', 'species', 'log10_total_fragments_count', 'cisTopic_nr_frag', 'cisTopic_log_nr_frag', 'pdf_values_for_fraction_of_fragments_in_peaks', 'pdf_values_for_duplication_ratio', 'nucleosome_signal', 'sample_id', 'duplication_ratio', 'log10_unique_fragments_in_peaks_count', 'duplication_count', 'log10_unique_fragments_count', 'unique_fragments_count', 'leiden', 'fraction_of_fragments_in_peaks', 'pycisTopic_leiden_10_1'
      var:	'Chromosome', 'Start', 'End', 'Width', 'cisTopic_nr_frag', 'cisTopic_log_nr_frag', 'cisTopic_nr_acc', 'cisTopic_log_nr_acc'
      obsm:	'UMAP'

In [8]:
tf_list = ['Sox11', 'Cebpb']

rna  = adata['scRNA']
mu_tg = np.array(rna[:,tg_list].X.mean(0)).reshape(-1)
mu_tf = np.array(rna[:,tf_list].X.mean(0)).reshape(-1)
print(mu_tf)

atac = adata['scATAC']
mu_re = np.array(atac[:,re_list].X.mean(0)).reshape(-1)
print(mu_re.shape)

[0.03820966 0.026773  ]
(41,)


In [9]:
import json 

tf_dic = {}
for i,tf in enumerate(tf_list):
    tf_dic[tf] = mu_tf[i].astype(float)

with open('tf_dic.json', 'w') as f:
    json.dump(tf_dic, f) 

tg_dic = {}
for i,tg in enumerate(tg_list):
    tg_dic[tg] = mu_tg[i].astype(float)

with open('tg_dic.json', 'w') as f:
    json.dump(tg_dic, f)

re_dic = {}

for i,re in enumerate(re_list):
    re_dic[re] = mu_re[i].astype(float)
with open('re_dic.json', 'w') as f:
    json.dump(re_dic, f)

In [10]:
with open('reg.txt', 'w', encoding = 'utf-8') as f:
    for line in reg_list:
        f.write(line + '\n')